# Generate Embeddings — `muskannnnn/Prototype` (Hugging Face)

Encodes the 2,130-product catalogue (`final_products.csv`) with the fine-tuned
embedding model, hosted at
[huggingface.co/muskannnnn/Prototype](https://huggingface.co/muskannnnn/Prototype),
and saves two files that `retrieval.py` loads directly:

- `dataset_embeddings.npy` — one row per product, L2-normalized so dot
  product = cosine similarity
- `dataset_embedding_ids.json` — the `sku` for each row, in the exact same
  order as the embeddings — this is what lets `retrieval.py` map a similarity
  score back to the right product (`embed_id_to_idx` in `retrieval.py`)

Requires a GPU (**Settings → Accelerator → GPU**, T4 x2 or P100) — encoding
2,130 short strings is fast on GPU and painfully slow on CPU.

Text sent to the model is deliberately **concise** — `name | Brand | Category`
only, no `description` — this is a different (shorter) text than the one
`chunking.py` builds for BM25, on purpose: a long, mixed-signal description
dilutes the embedding, while BM25 benefits from every extra word. Don't
"fix" this to match chunking.py's text — it's intentional.

## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers

## 2. Load the fine-tuned model from the Hugging Face Hub

`SentenceTransformer(repo_id, device=...)` downloads and caches the model the
first time this runs — no local checkpoint needed. This is the same repo id
`config.EMBEDDING_MODEL_NAME` in the API points at, so query-time encoding
and this offline corpus encoding stay in the same embedding space.

In [ ]:
import json
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# 1. Verify GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device.upper()}")
if device == "cpu":
    print("WARNING: no GPU detected. Turn on Settings -> Accelerator -> GPU "
          "before running this, or this cell will be extremely slow.")

# 2. Load the fine-tuned model straight from the Hub
MODEL_REPO_ID = "muskannnnn/Prototype"
model = SentenceTransformer(MODEL_REPO_ID, device=device)
print(f"Loaded {MODEL_REPO_ID} - embedding dim: {model.get_sentence_embedding_dimension()}")

## 3. Load the catalogue

Point `CSV_PATH` at wherever `final_products.csv` lands under `/kaggle/input/`
once you've added it as a Kaggle Dataset.

In [ ]:
CSV_PATH = "/kaggle/input/<your-dataset-name>/final_products.csv"

df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Loaded {len(df):,} products")

assert "sku" in df.columns, "final_products.csv must have a sku column - chunking.py uses it as the chunk id"
assert df["sku"].is_unique, "sku column has duplicate values - ids will not align with chunks.jsonl"

## 4. Build the text to embed

Same formatting as the working prototype run: `name | Brand: ... | Category: ...`,
no description. Missing `name`/`brand`/`category_hierarchy` values are filled
with empty strings rather than dropped, so every row still gets an embedding
(and `category_hierarchy`'s `>` separators become spaces so each level is a
plain searchable term, matching how `chunking.py` treats the same field).

In [ ]:
df["name"] = df["name"].fillna("")
df["brand"] = df["brand"].fillna("")
df["category_hierarchy"] = df["category_hierarchy"].fillna("").apply(lambda x: str(x).replace(">", " "))

combined_texts = (
    df["name"].astype(str) +
    " | Brand: " + df["brand"].astype(str) +
    " | Category: " + df["category_hierarchy"].astype(str)
).tolist()

print("--- Sample formatted text ---")
print(combined_texts[0])
print("------------------------------")

## 5. Encode

In [ ]:
print("Generating embeddings...")
embeddings = model.encode(
    combined_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # L2-normalize so dot product = cosine similarity
).astype(np.float32)

print(f"Embeddings generated on {device.upper()}! Shape: {embeddings.shape}")

## 6. Save embeddings + aligned IDs

This is the part the original notebook was missing — `retrieval.py` needs
`embedding_ids.json` positionally aligned with `embeddings.npy` to know which
product each row belongs to. `ids` is built from the same `df`, in the same
row order used to build `combined_texts`, so the alignment holds by
construction.

In [ ]:
ids = df["sku"].astype(str).tolist()

assert len(ids) == embeddings.shape[0], "ids/embeddings length mismatch - something reordered df between text-building and here"

np.save("dataset_embeddings.npy", embeddings)
with open("dataset_embedding_ids.json", "w", encoding="utf-8") as f:
    json.dump(ids, f, ensure_ascii=False)

print(f"Saved dataset_embeddings.npy {embeddings.shape} and dataset_embedding_ids.json ({len(ids)} ids)")

## 7. Next step

Download both files from this notebook's **Output** panel and place them at:

```
indexes/prototype/embeddings.npy         <- dataset_embeddings.npy
indexes/prototype/embedding_ids.json     <- dataset_embedding_ids.json
```

(renaming as shown — `config.py` expects those exact filenames).